# RetinaSafe — Paper figure generation

**Goal:** load all 6 BRSET-related result JSONs and generate the four paper figures: per-label AUROC, per-label ECE, fairness-audit summary, and the drusens camera-bias case study.

**Inputs (Notebook Output tab — attach all that you have committed):**
1. `samarthmishra208/brset-baseline` (ResNet-50 supervised baseline)
2. `samarthmishra208/brset-fairness-audit` (the bootstrap-CI audit for ResNet-50)
3. `samarthmishra208/retfound-linear-probe-brset` (or whatever you named it)
4. `samarthmishra208/dinov2-linear-probe-brset`
5. `samarthmishra208/retfound-finetune-brset`
6. `samarthmishra208/dinov2-finetune-brset` (the v2 — with LR 1e-5)

**Settings:** No GPU needed, Internet off. Runtime ~30 seconds.

**Outputs:** PNG and PDF files in `/kaggle/working/figures/`. Download from the notebook's Output tab after committing.

## Cell 1 — Load all result JSONs

In [ ]:
import os, json, glob, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

OUT = pathlib.Path("/kaggle/working/figures")
OUT.mkdir(exist_ok=True)

DISEASE_COLS = [
    "diabetic_retinopathy", "macular_edema", "scar", "nevus", "amd",
    "vascular_occlusion", "hypertensive_retinopathy", "drusens",
    "hemorrhage", "myopic_fundus", "increased_cup_disc", "other",
]
SHORT_LABELS = {
    "diabetic_retinopathy": "DR",
    "macular_edema": "MacEd",
    "scar": "Scar",
    "nevus": "Nevus",
    "amd": "AMD",
    "vascular_occlusion": "VasOcc",
    "hypertensive_retinopathy": "HypRet",
    "drusens": "Drusens",
    "hemorrhage": "Hem",
    "myopic_fundus": "Myopic",
    "increased_cup_disc": "CupDisc",
    "other": "Other",
}

def find_json(pattern_dirs, fname):
    for d in pattern_dirs:
        hits = glob.glob(f"/kaggle/input/**/{d}/**/{fname}", recursive=True)
        if hits: return hits[0]
    return None

# Find each notebook's test_metrics.json
results = {}

# 1. ResNet-50 supervised baseline (its test_metrics.json has per-label AUROC but no per_label_ece)
hits = glob.glob("/kaggle/input/**/brset-baseline/**/test_metrics.json", recursive=True)
print("ResNet-50 baseline test_metrics.json:", hits)
if hits:
    with open(hits[0]) as f:
        results["resnet50_supervised"] = json.load(f)

# 2. RETFound linear probe
for slug_substr in ["retfound-linear-probe", "retfound_linear_probe"]:
    hits = glob.glob(f"/kaggle/input/**/{slug_substr}*/**/test_metrics.json", recursive=True)
    if hits:
        with open(hits[0]) as f:
            results["retfound_probe"] = json.load(f)
        print(f"RETFound probe ({hits[0]})"); break

# 3. DINOv2 linear probe
for slug_substr in ["dinov2-linear-probe", "dinov2_linear_probe"]:
    hits = glob.glob(f"/kaggle/input/**/{slug_substr}*/**/test_metrics.json", recursive=True)
    if hits:
        with open(hits[0]) as f:
            results["dinov2_probe"] = json.load(f)
        print(f"DINOv2 probe ({hits[0]})"); break

# 4. RETFound fine-tuned
for slug_substr in ["retfound-finetune", "retfound_finetune"]:
    hits = glob.glob(f"/kaggle/input/**/{slug_substr}*/**/test_metrics.json", recursive=True)
    if hits:
        with open(hits[0]) as f:
            results["retfound_ft"] = json.load(f)
        print(f"RETFound FT ({hits[0]})"); break

# 5. DINOv2 fine-tuned (v2)
for slug_substr in ["dinov2-finetune", "dinov2_finetune"]:
    hits = glob.glob(f"/kaggle/input/**/{slug_substr}*/**/test_metrics.json", recursive=True)
    if hits:
        # Prefer v2 if multiple are present
        v2_hits = [h for h in hits if "v2" in h.lower()]
        h = v2_hits[0] if v2_hits else hits[0]
        with open(h) as f:
            results["dinov2_ft"] = json.load(f)
        print(f"DINOv2 FT ({h})"); break

# 6. Fairness audit with CIs for ResNet-50 (separate notebook)
hits = glob.glob("/kaggle/input/**/fairness_audit_with_cis.json", recursive=True)
fairness_audit_resnet50 = None
if hits:
    with open(hits[0]) as f:
        fairness_audit_resnet50 = json.load(f)
    print(f"ResNet-50 fairness audit with CIs ({hits[0]})")

print(f"\nLoaded {len(results)} result conditions: {list(results.keys())}")

## Cell 2 — Figure 1: Per-label AUROC across all 5 methods

In [ ]:
METHOD_ORDER = [
    ("resnet50_supervised",  "ResNet-50 (supervised)",        "#3b82f6"),  # blue
    ("retfound_probe",       "RETFound (linear probe)",       "#ef4444"),  # red
    ("dinov2_probe",         "DINOv2 (linear probe)",         "#a855f7"),  # purple
    ("retfound_ft",          "RETFound (fine-tuned)",         "#f59e0b"),  # orange
    ("dinov2_ft",            "DINOv2 (fine-tuned, LR=1e-5)",  "#10b981"),  # green
]

# Pull per-label AUROC and CIs for each method
def get_per_label(res):
    out = {}
    if "per_label" not in res:
        # ResNet-50 baseline format: per-class AUROC dict under different key
        if "test_auroc_per_class" in res:
            for d, v in res["test_auroc_per_class"].items():
                out[d] = {"auroc": v, "auroc_ci95": [v, v]}  # no CIs in old format
            return out
    for d, v in res["per_label"].items():
        out[d] = {"auroc": v["auroc"], "auroc_ci95": v.get("auroc_ci95", [v["auroc"], v["auroc"]])}
    return out

per_label_by_method = {k: get_per_label(results[k]) for k, _, _ in METHOD_ORDER if k in results}

# Make Figure 1
fig, ax = plt.subplots(figsize=(16, 7))
n_methods = len(per_label_by_method)
bar_width = 0.16
x = np.arange(len(DISEASE_COLS))
for i, (key, label, color) in enumerate([m for m in METHOD_ORDER if m[0] in per_label_by_method]):
    aurocs = [per_label_by_method[key].get(d, {}).get("auroc", np.nan) for d in DISEASE_COLS]
    cis    = [per_label_by_method[key].get(d, {}).get("auroc_ci95", [np.nan, np.nan]) for d in DISEASE_COLS]
    aurocs = np.array(aurocs, dtype=float)
    lo = np.array([c[0] for c in cis], dtype=float)
    hi = np.array([c[1] for c in cis], dtype=float)
    yerr = np.array([aurocs - lo, hi - aurocs])
    pos = x + (i - (n_methods - 1) / 2) * bar_width
    ax.bar(pos, aurocs, width=bar_width, label=label, color=color, alpha=0.85,
           yerr=yerr, capsize=2, error_kw={"elinewidth": 0.8, "ecolor": "#333"})

ax.set_xticks(x)
ax.set_xticklabels([SHORT_LABELS[d] for d in DISEASE_COLS], rotation=30, ha="right")
ax.set_ylabel("Test AUROC", fontsize=12)
ax.set_title("Per-label test AUROC across five methods on BRSET (with bootstrap 95% CIs)", fontsize=13)
ax.set_ylim(0.4, 1.0)
ax.axhline(0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6, label="random (0.5)")
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.legend(loc="upper left", bbox_to_anchor=(1.005, 1.0), fontsize=10, framealpha=0.95, borderaxespad=0)
plt.tight_layout()
plt.savefig(OUT / "figure1_per_label_auroc.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT / "figure1_per_label_auroc.pdf", bbox_inches="tight")
plt.show()
print(f"Saved {OUT/'figure1_per_label_auroc.png'}")

## Cell 3 — Figure 2: Per-label ECE comparison

In [ ]:
# ResNet-50 baseline has no per_label_ece — exclude it from this figure
ECE_METHOD_ORDER = [
    ("retfound_probe",   "RETFound (probe)",       "#ef4444"),
    ("dinov2_probe",     "DINOv2 (probe)",         "#a855f7"),
    ("retfound_ft",      "RETFound (FT)",          "#f59e0b"),
    ("dinov2_ft",        "DINOv2 (FT)",            "#10b981"),
]

fig, ax = plt.subplots(figsize=(14, 6))
n_methods = sum(1 for k, _, _ in ECE_METHOD_ORDER if k in results)
bar_width = 0.2
x = np.arange(len(DISEASE_COLS))

for i, (key, label, color) in enumerate([m for m in ECE_METHOD_ORDER if m[0] in results]):
    eces = [results[key].get("per_label_ece", {}).get(d, np.nan) for d in DISEASE_COLS]
    pos = x + (i - (n_methods - 1) / 2) * bar_width
    ax.bar(pos, eces, width=bar_width, label=label, color=color, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([SHORT_LABELS[d] for d in DISEASE_COLS], rotation=30, ha="right")
ax.set_ylabel("Expected Calibration Error (ECE)", fontsize=12)
ax.set_title("Per-label calibration (lower is better)", fontsize=13)
ax.set_ylim(0, max(0.5, max(
    max(results.get(k, {}).get("per_label_ece", {}).values() or [0])
    for k, _, _ in ECE_METHOD_ORDER if k in results
) * 1.1))
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.legend(loc="upper right", fontsize=10)
plt.tight_layout()
plt.savefig(OUT / "figure2_per_label_ece.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT / "figure2_per_label_ece.pdf", bbox_inches="tight")
plt.show()
print(f"Saved {OUT/'figure2_per_label_ece.png'}")

## Cell 4 — Figure 3: Fairness audit — count of significant subgroup gaps per method

Counts the number of (label × subgroup-axis) pairs where 95% CIs don't overlap. Lower = fairer.

In [ ]:
def count_significant_gaps(audit):
    # audit is a dict: {by_sex: [...], by_camera: [...], by_quality: [...], by_age_band: [...]}
    # Each value is a list of subgroup dicts with per_label results that have auroc_ci95
    counts = {"sex": 0, "camera": 0, "quality": 0, "age_band": 0}
    for axis_key, axis_name in [("by_sex", "sex"), ("by_camera", "camera"),
                                 ("by_quality", "quality"), ("by_age_band", "age_band")]:
        rows = audit.get(axis_key, [])
        if not rows: continue
        for d in DISEASE_COLS:
            ci_list = []
            for r in rows:
                v = r.get("per_label", {}).get(d)
                if v and v.get("auroc_ci95"):
                    ci_list.append(v["auroc_ci95"])
            for i in range(len(ci_list)):
                for j in range(i+1, len(ci_list)):
                    a, b = ci_list[i], ci_list[j]
                    if a[1] < b[0] or b[1] < a[0]:
                        counts[axis_name] += 1
    return counts

method_gap_counts = {}
for key, label, color in METHOD_ORDER:
    if key == "resnet50_supervised":
        # Use the separate fairness audit with CIs
        if fairness_audit_resnet50:
            method_gap_counts[key] = count_significant_gaps(fairness_audit_resnet50)
    elif key in results and "fairness_audit" in results[key]:
        method_gap_counts[key] = count_significant_gaps(results[key]["fairness_audit"])

print("Significant gap counts by method × axis:")
for key, counts in method_gap_counts.items():
    print(f"  {key:25s} {counts} (total={sum(counts.values())})")

# Stacked bar chart
fig, ax = plt.subplots(figsize=(11, 6))
methods = [k for k, _, _ in METHOD_ORDER if k in method_gap_counts]
labels  = [name for k, name, _ in METHOD_ORDER if k in method_gap_counts]
colors  = [c for k, _, c in METHOD_ORDER if k in method_gap_counts]

axes_keys = ["sex", "camera", "quality", "age_band"]
axes_colors = {"sex": "#fde047", "camera": "#fb923c", "quality": "#a78bfa", "age_band": "#34d399"}

bottoms = np.zeros(len(methods))
for axis in axes_keys:
    vals = np.array([method_gap_counts[m][axis] for m in methods])
    ax.bar(labels, vals, bottom=bottoms, label=f"by {axis}", color=axes_colors[axis], edgecolor="white", linewidth=0.5)
    bottoms += vals

# Annotate totals
for i, m in enumerate(methods):
    total = sum(method_gap_counts[m].values())
    ax.text(i, total + 0.15, str(total), ha="center", fontweight="bold", fontsize=11)

ax.set_ylabel("# statistically significant subgroup gaps\n(95% CIs do not overlap)", fontsize=11)
ax.set_title("Fairness audit summary across methods (lower is fairer)", fontsize=13)
ax.legend(loc="upper right", title="Demographic axis", fontsize=9)
ax.grid(axis="y", linestyle=":", alpha=0.3)
plt.setp(ax.get_xticklabels(), rotation=15, ha="right")
plt.tight_layout()
plt.savefig(OUT / "figure3_fairness_summary.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT / "figure3_fairness_summary.pdf", bbox_inches="tight")
plt.show()
print(f"\nSaved {OUT/'figure3_fairness_summary.png'}")

## Cell 5 — Figure 5: Drusens camera-bias case study (forest plot)

The headline fairness finding. Shows drusens AUROC ± 95% CI for Canon vs Nikon under each of the 5 methods.

In [ ]:
def get_drusens_camera_auroc(audit):
    out = {}  # camera -> (auroc, ci_lo, ci_hi)
    for r in audit.get("by_camera", []):
        cam = r["subgroup"].replace("camera=", "")
        v = r.get("per_label", {}).get("drusens")
        if v and v.get("auroc"):
            out[cam] = (v["auroc"], v["auroc_ci95"][0], v["auroc_ci95"][1])
    return out

drusens_data = {}
for key, label, color in METHOD_ORDER:
    if key == "resnet50_supervised":
        if fairness_audit_resnet50:
            drusens_data[key] = get_drusens_camera_auroc(fairness_audit_resnet50)
    elif key in results and "fairness_audit" in results[key]:
        drusens_data[key] = get_drusens_camera_auroc(results[key]["fairness_audit"])

# Forest plot
fig, ax = plt.subplots(figsize=(10, 6))
y_positions = []
y_labels = []
y_offset = 0

cameras = ["Canon CR", "NIKON NF5050"]
cam_colors = {"Canon CR": "#1f77b4", "NIKON NF5050": "#d62728"}

for key, method_label, _ in METHOD_ORDER:
    if key not in drusens_data: continue
    for j, cam in enumerate(cameras):
        if cam not in drusens_data[key]: continue
        auroc, lo, hi = drusens_data[key][cam]
        y = y_offset + 0.35 * j
        ax.errorbar(auroc, y, xerr=[[auroc - lo], [hi - auroc]],
                    fmt="o", color=cam_colors[cam], capsize=4, markersize=8, linewidth=2)
        ax.text(lo - 0.005, y, f"{auroc:.3f}", va="center", ha="right", fontsize=9, color=cam_colors[cam], fontweight="bold")
    y_labels.append(method_label)
    y_positions.append(y_offset + 0.175)
    y_offset += 1.2

ax.set_yticks(y_positions)
ax.set_yticklabels(y_labels, fontsize=11)
ax.set_xlabel("Drusens AUROC (per-camera subgroup, 95% CI)", fontsize=11)
ax.set_title("Drusens camera-bias case study: Canon vs Nikon across methods", fontsize=13)
ax.set_xlim(0.50, 1.0)
ax.grid(axis="x", linestyle=":", alpha=0.3)
ax.invert_yaxis()

# Legend
canon_patch = plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cam_colors["Canon CR"],
                         markersize=10, label="Canon CR")
nikon_patch = plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=cam_colors["NIKON NF5050"],
                         markersize=10, label="NIKON NF5050")
ax.legend(handles=[canon_patch, nikon_patch], loc="upper right", fontsize=10, framealpha=0.95)

plt.tight_layout()
plt.savefig(OUT / "figure5_drusens_camera_bias.png", dpi=200, bbox_inches="tight")
plt.savefig(OUT / "figure5_drusens_camera_bias.pdf", bbox_inches="tight")
plt.show()
print(f"Saved {OUT/'figure5_drusens_camera_bias.png'}")

## Cell 6 — Zip all figures + print summary

In [ ]:
import shutil

zip_path = pathlib.Path("/kaggle/working/paper_figures.zip")
if zip_path.exists(): zip_path.unlink()
shutil.make_archive(zip_path.with_suffix(""), "zip", root_dir=str(OUT))
print(f"Wrote {zip_path}  ({zip_path.stat().st_size / 1024:.1f} KB)")

print("\n=== Generated figures ===")
for p in sorted(OUT.glob("*")):
    print(f"  {p.name}")
print("\nDownload paper_figures.zip from the notebook's Output tab after committing.")